# Variant features

One row per lead variant of a qualifying disease credible set, aggregated over every credible
set that variant appears in: effect size, MAF, effective sample size, variance explained,
GERP constraint, VEP score, the concordance of effect direction across diseases, and the
diseases and therapeutic areas it is associated with. Methods "Variant-level pleiotropy
modelling".

These are the covariates the variant pleiotropy model is fitted on, and the source of the
directionality numbers and Supplementary Table 2.

Three definitions of lead_vPS and its concordance are carried side by side — the published
columns, the previous pass's `lead*` family, and the sign-gated `signedLead*` family that the
directionality analysis now reports on. The section below says what separates them.

Writes `variant_features`.

In [1]:
import pandas as pd
from gentropy.common.session import Session
from pyspark.sql import Window
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/22 17:11:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
credible_sets = session.spark.read.parquet(paper.derived("qualifying_credible_sets"))
# The variant-level analysis was built on the legacy hierarchy order, so the therapeutic-area
# count here uses that column (see 03_therapeutic_areas).
studies = session.spark.read.parquet(paper.derived("study_therapeutic_areas")).select(
    "studyId",
    f.col("mappedTherapeuticAreasLegacy").alias("mappedTherapeuticAreas"),
    "totalTherapeuticAreas",
    *paper.TA_COLUMNS.values(),
)
annotation = session.spark.read.parquet(paper.derived("study_annotation")).select(
    "studyId", "publicationDate", "effectiveSampleSize"
)
prioritised = (
    session.spark.read.parquet(paper.derived("prioritised_genes_per_cs"))
    .groupBy("studyLocusId")
    .agg(f.collect_list("geneId").alias("prioritisedGenes"))
)
variants = session.spark.read.parquet(paper.release("variant")).select(
    "variantId",
    f.filter("variantEffect", lambda x: x["method"] == "GERP")[0]["normalisedScore"].alias("gerpNormalised"),
    f.filter("variantEffect", lambda x: x["method"] == "VEP")[0]["score"].alias("vepScore"),
)
disease_names = session.spark.read.parquet(paper.release("disease") + "/disease.parquet").select("id", "name")

## Credible sets with their study covariates

In [3]:
rows = (
    credible_sets.join(prioritised, "studyLocusId", "left")
    .join(studies, "studyId", "inner")
    .join(annotation, "studyId", "inner")
    .withColumn("coefficientDetermination", f.col("variantStatistics.chi2Stat") / f.col("effectiveSampleSize"))
    .cache()
)
print("credible sets:", rows.count(), "| lead variants:", rows.select("variantId").distinct().count())

26/08/22 17:11:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


credible sets: 70618 | lead variants: 40706


## lead_vPS — the redefined variant pleiotropy score

`uniqueDiseases` counts every disease term any of the variant's credible sets carries, so a single
study annotated with several terms makes a variant look pleiotropic on one association and one
beta. `leadVPS` counts only **contributing** credible sets. A credible set contributes when

1. the variant is its lead variant (true of every row here by construction),
2. its study is mapped to **exactly one** disease term, and
3. `rescaledStatistics.directionOfEffect` is **non-null** — the sign gate, added 2026-08-21.

A variant with no contributing credible set has no `leadVPS` at all; the flag says so and no
sentinel value is written.

Direction comes from the harmonised effect-allele beta. `originalBeta` is the effect of the
alternate allele of the variant id, which is fixed across every credible set of a variant, so its
sign is comparable across studies without conversion; `rescaledStatistics.directionOfEffect` is
that sign, already computed. This replaces `rescaledStatistics.minorAlleleEstimatedBeta`, whose
minor-allele flip is keyed on the alternate-allele frequency **in the study's own major LD
population** and therefore reverses between studies of different ancestry whenever that frequency
straddles 0.5.

Where a disease is carried by more than one contributing study, the most significant association
by P value gives its direction. Concordance is the largest share of same-direction effects across
the variant's diseases.

### Two column families, both written

| family | condition (3), the sign gate | concordance |
| --- | --- | --- |
| `leadVPS`, `leadDirectionalConcordance` | **not** applied | 1 where `leadVPS` is 1, computed where at least two diseases carry a signed effect, null otherwise |
| `signedLeadVPS`, `signedLeadDirectionalConcordance` | applied | always defined where `signedLeadVPS` is, because every counted disease is signed by construction |

The first is the previous pass's definition, kept so its lineage stays reproducible. The second is
the amendment: a disease that carries no signed effect is not counted at all. Removing those
diseases from the count also removes the "fewer than two signed diseases" group, so the three
groups of the previous pass — defined, concordance-undefined, excluded — collapse to two.

**This is not a subtraction of the undefined group.** The gate also trims variants outside it: a
variant with five contributing diseases one of which is unsigned has `signedLeadVPS` 4, not 5. The
whole distribution is recomputed, never adjusted.

The published `betaSignConcordance` and `uniqueDiseases` are kept unchanged alongside both.

### What the sign gate costs, before it is applied

Two checks, over the credible sets whose study carries exactly one disease term. The first asks
whether `rescaledStatistics.directionOfEffect` and `originalBeta` are missing together — the gate
is only meaningful if the direction column is the same information as the beta. The second asks
where the missing directions sit, since the cost of the gate falls entirely on whatever kind of
study carries them.

In [4]:
single_disease = rows.filter(f.size("diseaseIds") == 1).cache()
direction = f.col("rescaledStatistics.directionOfEffect")
beta = f.col("originalBeta")

nulls = single_disease.select(
    f.count("*").alias("credibleSets"),
    f.sum(direction.isNull().cast("int")).alias("directionNull"),
    f.sum(beta.isNull().cast("int")).alias("betaNull"),
    f.sum((direction.isNull() & beta.isNotNull()).cast("int")).alias("directionNullBetaPresent"),
    f.sum((direction.isNotNull() & beta.isNull()).cast("int")).alias("directionPresentBetaNull"),
    f.sum((direction == 0).cast("int")).alias("directionZero"),
).toPandas()
print("credible sets of single-disease studies, and where a direction is missing:")
print(nulls.T.to_string(header=False))

# The two columns are the same information, so the gate loses nothing the beta would have kept.
# The one exception is a credible set whose beta is exactly 0: a signum has no sign there.
disagreement = single_disease.filter(direction.isNull() & beta.isNotNull()).select(
    "studyId",
    "variantId",
    "originalBeta",
    "originalStandardError",
    f.col("rescaledStatistics.absEstimatedBeta").alias("absEstimatedBeta"),
)
print(f"\ndirection missing where the beta is present: {disagreement.count()}")
disagreement.show(10, truncate=False)

credible sets of single-disease studies, and where a direction is missing:
credibleSets              65431
directionNull              5706
betaNull                   5705
directionNullBetaPresent      1
directionPresentBetaNull      0
directionZero                 0



direction missing where the beta is present: 1


+------------+--------------+------------+---------------------+--------------------+
|studyId     |variantId     |originalBeta|originalStandardError|absEstimatedBeta    |
+------------+--------------+------------+---------------------+--------------------+
|GCST90270934|X_15523993_G_A|0.0         |NULL                 |0.053930039584935044|
+------------+--------------+------------+---------------------+--------------------+



In [5]:
# Where the missing directions sit. `projectId` names the source and `hasSumstats` says whether
# harmonised summary statistics were ingested; a curated GWAS Catalog association without them
# carries an effect size only if the curator recorded one.
source = session.spark.read.parquet(paper.derived("study_annotation")).select("studyId", "projectId")
release_studies = session.spark.read.parquet(paper.release("study")).select("studyId", "hasSumstats")
by_source = (
    single_disease.join(source, "studyId", "left")
    .join(release_studies, "studyId", "left")
    .groupBy("projectId", "hasSumstats")
    .agg(
        f.count("*").alias("credibleSets"),
        f.sum(direction.isNull().cast("int")).alias("directionNull"),
        f.countDistinct("variantId").alias("leadVariants"),
        f.countDistinct(f.when(direction.isNull(), f.col("variantId"))).alias("leadVariantsAffected"),
        f.countDistinct("studyId").alias("studies"),
        f.countDistinct(f.when(direction.isNull(), f.col("studyId"))).alias("studiesAffected"),
    )
    .orderBy(f.col("directionNull").desc())
    .toPandas()
)
print(by_source.to_string(index=False))

# How far the gate reaches, per lead variant: the diseases it takes away, and the variants it
# leaves with nothing at all.
before = single_disease.select("variantId", f.col("diseaseIds")[0].alias("diseaseId")).distinct()
after = (
    single_disease.filter(direction.isNotNull())
    .select("variantId", f.col("diseaseIds")[0].alias("diseaseId"))
    .distinct()
)
impact = (
    before.groupBy("variantId")
    .agg(f.count("*").alias("before"))
    .join(after.groupBy("variantId").agg(f.count("*").alias("after")), "variantId", "left")
    .fillna({"after": 0})
    .select(
        f.count("*").alias("variantsWithAContributingCredibleSet"),
        f.sum((f.col("after") == 0).cast("int")).alias("variantsLosingEveryDisease"),
        f.sum(((f.col("after") > 0) & (f.col("after") < f.col("before"))).cast("int")).alias("variantsTrimmed"),
        f.sum((f.col("after") == f.col("before")).cast("int")).alias("variantsUntouched"),
        f.sum("before").alias("diseaseTermsBefore"),
        f.sum("after").alias("diseaseTermsAfter"),
    )
    .toPandas()
)
print()
print(impact.T.to_string(header=False))

  projectId hasSumstats  credibleSets  directionNull  leadVariants  leadVariantsAffected  studies  studiesAffected
       GCST       False         22759           5687         14985                  4422     2381             1004
       GCST        None            31             19            29                    18        6                4
       GCST        True         29402              0         19569                     0     1850                0
FINNGEN_R12        True         13239              0          9255                     0      728                0



variantsWithAContributingCredibleSet  38273
variantsLosingEveryDisease             2801
variantsTrimmed                         675
variantsUntouched                     34797
diseaseTermsBefore                    50519
diseaseTermsAfter                     46816


In [6]:
# One row per contributing credible set: the study carries exactly one disease term.
contributing = rows.filter(f.size("diseaseIds") == 1).withColumn("diseaseId", f.col("diseaseIds")[0])
# The sign gate: a credible set with no direction of effect cannot sign a disease, so it does not
# contribute at all. Applied to the second column family only.
signed_contributing = contributing.filter(f.col("rescaledStatistics.directionOfEffect").isNotNull())

# Direction per disease, from the harmonised effect-allele beta with no allele conversion. Ties on
# P value are broken by studyLocusId so the pick is reproducible.
most_significant = Window.partitionBy("variantId", "diseaseId").orderBy(
    f.col("variantStatistics.pValueExponent").asc(),
    f.col("variantStatistics.pValueMantissa").asc(),
    f.col("studyLocusId").asc(),
)


def lead_pleiotropy_of(frame, prefix):
    """lead_vPS, its direction counts, its context and its concordance over one contributing set.

    `prefix` names the column family: `lead` for the previous definition, `signedLead` for the
    amended one, whose input frame has already been gated on a non-null direction of effect. The
    concordance expression is the same for both — under the gate its null branch is unreachable,
    because every counted disease carries a signed effect.
    """
    per_disease = (
        frame.withColumn("rank", f.row_number().over(most_significant))
        .filter(f.col("rank") == 1)
        .select(
            "variantId",
            "diseaseId",
            f.col("rescaledStatistics.directionOfEffect").alias("direction"),
            f.col("studyId").alias("contributingStudyId"),
        )
    )
    directions = per_disease.groupBy("variantId").agg(
        f.count("*").alias(f"{prefix}VPS"),
        f.countDistinct("contributingStudyId").alias(f"{prefix}ContributingStudies"),
        f.sum(f.when(f.col("direction").isNotNull(), 1).otherwise(0)).alias(f"{prefix}SignedDiseases"),
        f.sum(f.when(f.col("direction") > 0, 1).otherwise(0)).alias(f"{prefix}UpDiseases"),
        f.sum(f.when(f.col("direction") < 0, 1).otherwise(0)).alias(f"{prefix}DownDiseases"),
    )
    # The therapeutic areas of the contributing diseases, on the same legacy hierarchy column the
    # published variant-level area count uses. A contributing study carries one disease term, so
    # its area set is that disease's.
    context = frame.groupBy("variantId").agg(
        f.array_sort(f.array_distinct(f.collect_list("diseaseId"))).alias(f"{prefix}DiseaseIds"),
        f.array_sort(f.array_distinct(f.flatten(f.collect_list("mappedTherapeuticAreas")))).alias(
            f"{prefix}TherapeuticAreas"
        ),
    )
    table = (
        directions.join(context, "variantId", "inner")
        .withColumn(f"{prefix}UniqueTherapeuticAreas", f.size(f"{prefix}TherapeuticAreas"))
        .withColumn(
            f"{prefix}DirectionalConcordance",
            f.when(f.col(f"{prefix}VPS") == 1, f.lit(1.0))
            .when(
                f.col(f"{prefix}SignedDiseases") >= 2,
                f.greatest(f.col(f"{prefix}UpDiseases"), f.col(f"{prefix}DownDiseases"))
                / f.col(f"{prefix}SignedDiseases"),
            )
            .otherwise(f.lit(None).cast("double")),
        )
        .cache()
    )
    assert table.filter(f.col(f"{prefix}VPS") != f.size(f"{prefix}DiseaseIds")).count() == 0
    return table


lead_pleiotropy = lead_pleiotropy_of(contributing, "lead")
signed_lead_pleiotropy = lead_pleiotropy_of(signed_contributing, "signedLead")

# Under the gate every counted disease is signed, so the count and the signed count agree and the
# concordance is never null. Both are asserted rather than assumed.
assert signed_lead_pleiotropy.filter(f.col("signedLeadSignedDiseases") != f.col("signedLeadVPS")).count() == 0
assert signed_lead_pleiotropy.filter(f.col("signedLeadDirectionalConcordance").isNull()).count() == 0

lead_variants = rows.select("variantId").distinct().count()
print("variants with a contributing credible set:", lead_pleiotropy.count(), "| of", lead_variants)
print("under the sign gate:", signed_lead_pleiotropy.count(), "| of", lead_variants)

variants with a contributing credible set: 38273 | of 40706


under the sign gate: 35472 | of 40706


## Aggregate to variants

`betaSignConcordance` is the share of associations agreeing on the more common direction of
effect, computed on the minor allele so that direction is comparable across studies.

In [7]:
minor_beta = f.col("rescaledStatistics.minorAlleleEstimatedBeta")
# Effect size is the unsigned rescaled beta; the minor-allele beta is used only for direction,
# where the sign has to be comparable across studies. The published Figure 3 covariate is the
# unsigned one (verified against the table the figure was built from).
abs_beta = f.abs(f.col("rescaledStatistics.absEstimatedBeta"))
has_beta = f.col("originalBeta").isNotNull()
positive = f.sum(f.when(has_beta & (minor_beta > 0), 1.0).otherwise(0.0))
negative = f.sum(f.when(has_beta & (minor_beta < 0), 1.0).otherwise(0.0))
with_beta = f.sum(f.when(has_beta, 1.0).otherwise(0.0))

features = (
    rows.groupBy("variantId")
    .agg(
        f.mean(f.when(f.col("majorLdPopulation.ldPopulation") == "nfe", 1).otherwise(0)).alias("nonEURProportion"),
        f.min(abs_beta).alias("minAbsBeta"),
        f.max(abs_beta).alias("maxAbsBeta"),
        f.min("coefficientDetermination").alias("minCoefficientDetermination"),
        f.max("coefficientDetermination").alias("maxCoefficientDetermination"),
        f.min("effectiveSampleSize").alias("minEffectiveSampleSize"),
        f.max("effectiveSampleSize").alias("maxEffectiveSampleSize"),
        f.min("rescaledStatistics.varG").alias("minVarG"),
        f.max("rescaledStatistics.varG").alias("maxVarG"),
        f.max("majorLdPopulationMaf.value").alias("maxMAF"),
        f.avg(f.when(has_beta, f.signum(minor_beta))).alias("averageBetaSign"),
        f.stddev_pop(f.when(has_beta, f.signum(minor_beta))).alias("stdBetaSign"),
        f.greatest(positive / with_beta, negative / with_beta).alias("betaSignConcordance"),
        f.min("publicationDate").alias("earliestPublicationDate"),
        f.array_distinct(f.flatten(f.collect_list("diseaseIds"))).alias("diseaseIds"),
        f.size(f.array_distinct(f.flatten(f.collect_list("diseaseIds")))).alias("uniqueDiseases"),
        f.array_distinct(f.flatten(f.collect_list("mappedTherapeuticAreas"))).alias("therapeuticAreas"),
        f.size(f.array_distinct(f.flatten(f.collect_list("mappedTherapeuticAreas")))).alias("uniqueTherapeuticAreas"),
        f.array_distinct(f.flatten(f.collect_list("prioritisedGenes"))).alias("prioritisedGenes"),
        f.countDistinct("studyId").alias("totalStudies"),
        *[f.sum(column).alias(column) for column in paper.TA_COLUMNS.values()],
    )
    .join(variants, "variantId", "left")
    .join(lead_pleiotropy, "variantId", "left")
    .join(signed_lead_pleiotropy, "variantId", "left")
    # An explicit flag rather than a sentinel: a variant with no contributing credible set has no
    # lead_vPS and no concordance, and is excluded from the directionality analysis downstream.
    .withColumn("leadVPSDefined", f.col("leadVPS").isNotNull())
    .withColumn("leadConcordanceDefined", f.col("leadDirectionalConcordance").isNotNull())
    # Under the sign gate the two flags are one: a variant either has a signed contributing
    # credible set, in which case both its lead_vPS and its concordance are defined, or it has
    # none and neither is.
    .withColumn("signedLeadVPSDefined", f.col("signedLeadVPS").isNotNull())
)
features.write.mode("overwrite").parquet(paper.derived("variant_features"))

features = session.spark.read.parquet(paper.derived("variant_features"))
print("lead variants:", features.count())
assert (
    features.filter(f.col("signedLeadVPSDefined") != f.col("signedLeadDirectionalConcordance").isNotNull()).count() == 0
)

lead variants: 40706


In [8]:
# How the three definitions compare, over every lead variant.
summary = features.select(
    f.count("*").alias("leadVariants"),
    f.sum((f.col("uniqueDiseases") > 1).cast("int")).alias("published: uniqueDiseases > 1"),
    f.sum(f.col("betaSignConcordance").isNotNull().cast("int")).alias("published: concordance defined"),
    f.sum(f.col("leadVPSDefined").cast("int")).alias("ungated: leadVPS defined"),
    f.sum((~f.col("leadVPSDefined")).cast("int")).alias("ungated: leadVPS undefined"),
    f.sum((f.col("leadVPS") > 1).cast("int")).alias("ungated: leadVPS > 1"),
    f.sum(f.col("leadConcordanceDefined").cast("int")).alias("ungated: concordance defined"),
    f.sum(f.col("signedLeadVPSDefined").cast("int")).alias("gated: signedLeadVPS defined"),
    f.sum((~f.col("signedLeadVPSDefined")).cast("int")).alias("gated: signedLeadVPS undefined"),
    f.sum((f.col("signedLeadVPS") > 1).cast("int")).alias("gated: signedLeadVPS > 1"),
).toPandas()
print(summary.T.to_string(header=False))

# vPS keeps counting every disease term, so uniqueDiseases must be unchanged; leadVPS can only be
# smaller, since it drops the terms that only ever arrive alongside another in the same study, and
# signedLeadVPS smaller again, since it drops the terms that carry no signed effect.
comparison = features.select(
    f.sum((f.col("leadVPS") > f.col("uniqueDiseases")).cast("int")).alias("leadVPS above uniqueDiseases"),
    f.sum((f.col("leadVPS") < f.col("uniqueDiseases")).cast("int")).alias("leadVPS below uniqueDiseases"),
    f.sum((f.col("leadVPS") == f.col("uniqueDiseases")).cast("int")).alias("leadVPS equal"),
    f.sum((f.col("signedLeadVPS") > f.col("leadVPS")).cast("int")).alias("signedLeadVPS above leadVPS"),
    f.sum((f.col("signedLeadVPS") < f.col("leadVPS")).cast("int")).alias("signedLeadVPS below leadVPS"),
    f.sum((f.col("signedLeadVPS") == f.col("leadVPS")).cast("int")).alias("signedLeadVPS equal"),
).toPandas()
print()
print(comparison.T.to_string(header=False))
assert int(comparison["leadVPS above uniqueDiseases"].iloc[0]) == 0
assert int(comparison["signedLeadVPS above leadVPS"].iloc[0]) == 0

# The distribution of each score, since the gate recomputes it rather than adjusting it.
distribution = features.select("uniqueDiseases", "leadVPS", "signedLeadVPS").toPandas().describe().round(3)
print()
print(distribution.to_string())

leadVariants                    40706
published: uniqueDiseases > 1    9828
published: concordance defined  37219
ungated: leadVPS defined        38273
ungated: leadVPS undefined       2433
ungated: leadVPS > 1             6383
ungated: concordance defined    37667
gated: signedLeadVPS defined    35472
gated: signedLeadVPS undefined   5234
gated: signedLeadVPS > 1         5919

leadVPS above uniqueDiseases      0
leadVPS below uniqueDiseases   1845
leadVPS equal                 36428
signedLeadVPS above leadVPS       0
signedLeadVPS below leadVPS     675
signedLeadVPS equal           34797



       uniqueDiseases    leadVPS  signedLeadVPS
count       40706.000  38273.000      35472.000
mean            1.480      1.320          1.320
std             1.544      1.268          1.268
min             1.000      1.000          1.000
25%             1.000      1.000          1.000
50%             1.000      1.000          1.000
75%             1.000      1.000          1.000
max            85.000     71.000         71.000


### Retention at the top of the distribution

For the response letter: of the lead variants the published definition calls highly pleiotropic —
`uniqueDiseases >= 10` — how many survive each redefinition, and by how much their disease count
shrinks. Retention is `newCount / uniqueDiseases` per variant.

In [9]:
counts = features.select("variantId", "uniqueDiseases", "leadVPS", "signedLeadVPS").toPandas()
high = counts[counts["uniqueDiseases"] >= 10].copy()

records = []
for label, column in [("ungated leadVPS", "leadVPS"), ("gated signedLeadVPS", "signedLeadVPS")]:
    kept = high[column].fillna(0)
    records.append(
        {
            "definition": label,
            "published >= 10 diseases": len(high),
            "still >= 10": int((kept >= 10).sum()),
            "still > 1": int((kept > 1).sum()),
            "fallen to 0": int((kept == 0).sum()),
            "median retention (%)": round(100 * float((kept / high["uniqueDiseases"]).median()), 1),
            "mean retention (%)": round(100 * float((kept / high["uniqueDiseases"]).mean()), 1),
        }
    )
print(pd.DataFrame(records).to_string(index=False))

         definition  published >= 10 diseases  still >= 10  still > 1  fallen to 0  median retention (%)  mean retention (%)
    ungated leadVPS                       197          123        182           11                  83.3                74.1
gated signedLeadVPS                       197          118        182           11                  80.0                70.9


## Cross-check against the table the published Figure 3 was built from

In [10]:
columns = ["variantId", "maxAbsBeta", "maxMAF", "maxEffectiveSampleSize", "maxVarG", "gerpNormalised", "vepScore"]
new = features.select(columns).toPandas().drop_duplicates("variantId")
published = (
    session.spark.read.parquet(
        str(paper.ROOT / "chapters/_legacy/03-manuscript-figures/figure_3/python_scripts/variant_pleiotropy")
    )
    .select(columns)
    .toPandas()
    .drop_duplicates("variantId")
)
merged = new.merge(published, on="variantId", how="outer", suffixes=("_new", "_published"), indicator=True)
print(merged["_merge"].value_counts().to_string())
both = merged[merged["_merge"] == "both"]
for column in columns[1:]:
    delta = (both[f"{column}_new"] - both[f"{column}_published"]).abs().max()
    print(f"{column:<28} max abs difference: {delta:.3e}")

_merge
both          40706
left_only         0
right_only        0
maxAbsBeta                   max abs difference: 4.441e-16
maxMAF                       max abs difference: 0.000e+00
maxEffectiveSampleSize       max abs difference: 0.000e+00
maxVarG                      max abs difference: 0.000e+00
gerpNormalised               max abs difference: 0.000e+00
vepScore                     max abs difference: 0.000e+00
